In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
data_path = '../data/'

orders = pd.read_csv(data_path + 'olist_orders_dataset.csv')
customers = pd.read_csv(data_path + 'olist_customers_dataset.csv')
order_items = pd.read_csv(data_path + 'olist_order_items_dataset.csv')
order_payments = pd.read_csv(data_path + 'olist_order_payments_dataset.csv')
order_reviews = pd.read_csv(data_path + 'olist_order_reviews_dataset.csv')
products = pd.read_csv(data_path + 'olist_products_dataset.csv')
sellers = pd.read_csv(data_path + 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(data_path + 'product_category_name_translation.csv')
geolocation = pd.read_csv(data_path + 'olist_geolocation_dataset.csv')

In [4]:
datasets = {
    'orders': orders,
    'customers': customers,
    'order_items': order_items,
    'order_payments': order_payments,
    'order_reviews': order_reviews,
    'products': products,
    'sellers': sellers,
    'category_translation': category_translation,
    'geolocation': geolocation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape} | nulls: {df.isnull().sum().sum()}")

orders: (99441, 8) | nulls: 4908
customers: (99441, 5) | nulls: 0
order_items: (112650, 7) | nulls: 0
order_payments: (103886, 5) | nulls: 0
order_reviews: (99224, 7) | nulls: 145903
products: (32951, 9) | nulls: 2448
sellers: (3095, 4) | nulls: 0
category_translation: (71, 2) | nulls: 0
geolocation: (1000163, 5) | nulls: 0


In [5]:
print(orders.isnull().sum())
print("\nOrder status distribution:")
print(orders['order_status'].value_counts())

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order status distribution:
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: order_status, dtype: int64


Nulls make complete sense now:

order_approved_at — 160 nulls → cancelled/created orders never got approved
order_delivered_carrier_date — 1,783 nulls → not yet shipped
order_delivered_customer_date — 2,965 nulls → not yet delivered

96,478 delivered out of 99,441 total = 97% delivery completion rate

In [10]:
datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at', 
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Date range of orders:")
print(f"First order: {orders['order_purchase_timestamp'].min()}")
print(f"Last order: {orders['order_purchase_timestamp'].max()}")
print(f"\nTotal delivered orders: {(orders['order_status'] == 'delivered').sum()}")

Date range of orders:
First order: 2016-09-04 21:15:19
Last order: 2018-10-17 17:30:18

Total delivered orders: 96478


In [11]:
delivered = orders[orders['order_status'] == 'delivered'].copy()

delivered['delay_days'] = (
    delivered['order_delivered_customer_date'] - 
    delivered['order_estimated_delivery_date']
).dt.days

print(f"On-time deliveries: {(delivered['delay_days'] <= 0).sum()}")
print(f"Late deliveries: {(delivered['delay_days'] > 0).sum()}")
print(f"Late delivery rate: {(delivered['delay_days'] > 0).mean()*100:.1f}%")
print(f"\nAvg delay for late orders: {delivered[delivered['delay_days'] > 0]['delay_days'].mean():.1f} days")

On-time deliveries: 89936
Late deliveries: 6534
Late delivery rate: 6.8%

Avg delay for late orders: 10.6 days


6,534 late orders out of 96,478 delivered
Every late order averages 10.6 days past promised date

That's not catastrophic — but 10.6 days late is a terrible customer experience. This will show up in review scores

In [12]:
print(order_reviews.isnull().sum())
print(f"\nReview score distribution:")
print(order_reviews['review_score'].value_counts().sort_index())

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Review score distribution:
1    11424
2     3151
3     8179
4    19142
5    57328
Name: review_score, dtype: int64


**5 star:  57,328  →  57.8%
4 star:  19,142  →  19.3%
3 star:   8,179  →   8.2%
2 star:   3,151  →   3.2%
1 star:  11,424  →  11.5%**





Notice something interesting — 1 star (11.5%) is higher than 2+3 star combined (11.4%). Customers either love it or hate it. No middle ground. This is a story worth telling in your README.
77.1% of customers rate 4 or 5 stars — overall satisfaction is high. Which means the unhappy 11.5% are likely driven by something specific. Your delivery delay analysis will probably explain a big chunk of it.

In [13]:
# Merge English category names
products = products.merge(
    category_translation, 
    on='product_category_name', 
    how='left'
)

# Fill missing category with 'unknown'
products['product_category_name_english'] = (
    products['product_category_name_english']
    .fillna('unknown')
)

print(f"Products with unknown category: {(products['product_category_name_english'] == 'unknown').sum()}")
print(f"\nTop 10 categories:")
print(products['product_category_name_english'].value_counts().head(10))

Products with unknown category: 623

Top 10 categories:
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: product_category_name_english, dtype: int64


In [14]:
df = orders.merge(customers, on='customer_id', how='left')
df = df.merge(order_items, on='order_id', how='left')
df = df.merge(order_payments, on='order_id', how='left')
df = df.merge(products, on='product_id', how='left')
df = df.merge(order_reviews, on='order_id', how='left')

print(f"Master dataframe shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

Master dataframe shape: (119143, 37)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


In [15]:
# Check for duplicate order_ids in master df
print(f"Unique orders in master df: {df['order_id'].nunique()}")
print(f"Unique customers: {df['customer_unique_id'].nunique()}")
print(f"Unique products: {df['product_id'].nunique()}")

# Quick revenue sanity check
delivered_df = df[df['order_status'] == 'delivered']
print(f"\nTotal revenue (delivered orders): R${delivered_df['payment_value'].sum():,.2f}")
print(f"Average order value: R${delivered_df.groupby('order_id')['payment_value'].sum().mean():,.2f}")

Unique orders in master df: 99441
Unique customers: 96096
Unique products: 32951

Total revenue (delivered orders): R$19,881,945.07
Average order value: R$206.08


## Day 1 Summary

**Data loaded:** 9 tables, merged into master dataframe (119,143 rows, 37 columns)

**Key observations:**
- 99,441 total orders | 96,478 delivered (97.0%)
- Late delivery rate: 6.8% | Avg delay: 10.6 days when late
- Review scores: 77.1% positive (4–5 star) | 11.5% strongly negative (1 star)
- 1-star rate higher than 2+3 star combined — suggests specific failure points
- Top categories: bed_bath_table, sports_leisure, furniture_decor

**Hypothesis check:**
Revenue plateau likely driven by repeat purchase failure, not acquisition.
Delivery delays may explain 1-star concentration. To be proven in SQL analysis.

**Next:** SQL analysis — funnel drop-off, revenue trends, delivery vs satisfaction